# OpenAI and SAELens sparse features

Install `.[sae,notebooks]`. Uses the actual vendored OpenAI inference module and SAELens JumpReLU, not copies of the full training stacks. Train a tiny synthetic OpenAI SAE; inspect features, reconstruction and error-preserving edits. The SAELens fixture validates the architecture/API used for Scope-style dictionaries, NOT downloaded Gemma Scope 2 weights.

Run all cells in a fresh kernel. The final cell records local runtime and kernel RSS.

In [1]:
import os, time, json, resource, sys
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["USE_TF"] = "0"
os.environ["WANDB_MODE"] = "disabled"
import torch, numpy as np
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.manual_seed(7)
np.random.seed(7)
started = time.perf_counter()
print("CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads")


CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads


In [2]:
import tempfile
from autoexplain._vendor.openai_sae import Autoencoder
from autoexplain.features import SAEAdapter, load_saelens_local, feature_examples
from sae_lens.saes.jumprelu_sae import JumpReLUSAE, JumpReLUSAEConfig
data = torch.rand(96, 4)
data[data < .7] = 0
sae = Autoencoder(n_latents=12, n_inputs=4)
opt = torch.optim.Adam(sae.parameters(), lr=.03)
for _ in range(80):
    z, context = sae.encode(data)
    reconstruction = sae.decode(z, context)
    loss = (reconstruction-data).square().mean() + .01*z.abs().mean()
    opt.zero_grad(); loss.backward(); opt.step()
adapter = SAEAdapter(sae, backend="openai")
heldout = torch.rand(8, 4)
report = adapter.inspect(heldout, top_k=3)
print("Held-out MSE:", report["reconstruction_mse"], "mean L0:", report["mean_l0"])
assert torch.allclose(adapter.edit(heldout, feature=0, scale=1), heldout, atol=1e-6)
changed = adapter.edit(heldout, feature=0, scale=0)
print("Feature-zero ablation change:", (changed-heldout).norm().item())
print(feature_examples(report["features"], feature=0, k=3))
jump = JumpReLUSAE(JumpReLUSAEConfig(d_in=4, d_sae=12))
with tempfile.TemporaryDirectory() as directory:
    jump.save_model(directory)
    local = load_saelens_local(directory)
    assert torch.equal(jump.encode(heldout), local.encode(heldout)[0])
print("JumpReLU save/load passed. Real pretrained Scope artifacts require separate access and testing.")


Held-out MSE: 0.0010647819144651294 mean L0: 6.75
Feature-zero ablation change: 1.586154818534851
{'flat_indices': tensor([6, 3, 4]), 'scores': tensor([1.0024, 0.9106, 0.7660]), 'position_shape': torch.Size([8])}
JumpReLU save/load passed. Real pretrained Scope artifacts require separate access and testing.


In [3]:
elapsed = time.perf_counter() - started
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
rss_mib = rss / (1024**2 if sys.platform == "darwin" else 1024)
print(json.dumps({"tutorial_compute_wall_seconds": round(elapsed, 3),
                 "kernel_lifetime_peak_rss_mib": round(rss_mib, 2),
                 "memory_scope": "kernel only, including imports; not aggregate system memory"}))


{"tutorial_compute_wall_seconds": 3.708, "kernel_lifetime_peak_rss_mib": 648.9, "memory_scope": "kernel only, including imports; not aggregate system memory"}
